In [1]:
!pip3 install -q openai #gpt-4o-mini-2024-07-18


[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import json

In [3]:
from openai import OpenAI


def gpt_stage1(query,bg, model, prompt_path):
    
    client = OpenAI(api_key = "...")
    mes = ''
    with open (prompt_path, 'r') as file:
        for line in file:
            mes += line
    #print(mes)
    #print(bg)
    

    completion = client.chat.completions.create(
      model= model,
      #model="gpt-3.5-turbo",
      messages = [
            {
               "role": "system",
               "content": mes
            },
            {"role": "user", "content":f"original query: {query}, background information: {bg}"}],
      max_tokens=1024,
      #top_p=1,
      #temperature=0
    )


    return completion.choices[0].message.content

def gpt_stage2(query, kp, model, prompt_path):
    
    client = OpenAI(api_key = "...")
    mes = ''
    with open (prompt_path, 'r') as file:
        for line in file:
            mes += line
    #print(mes)
    

    completion = client.chat.completions.create(
      model= model,
      #model="gpt-3.5-turbo",
      messages = [
            {
               "role": "system",
               "content": mes
            },
            {"role": "user", "content":f"original query: {query}; extracted key points: {kp}"}],
      max_tokens=1024,
      #top_p=1,
      #temperature=1.0
    )


    return completion.choices[0].message.content

def gpt_stage3(conv, model, prompt_path):
    
    client = OpenAI(api_key = "...")
    mes = ''
    with open (prompt_path, 'r') as file:
        for line in file:
            mes += line
    #print(mes)
    

    completion = client.chat.completions.create(
      model= model,
      #model="gpt-3.5-turbo",
      messages = [
            {
               "role": "system",
               "content": mes
            },

            {"role": "user", "content":f"conversation: {conv}"}],
      max_tokens=1024,
      #top_p=1,
      #temperature=1.0
    )


    return completion.choices[0].message.content

In [4]:
# method 3, directly let llm generate conversations
from openai import OpenAI


def gpt_conv(query, bg, model):
    client = OpenAI(api_key = "...")


    completion = client.chat.completions.create(
      model=model,
      #model="gpt-3.5-turbo",
      messages = [
            {
               "role": "user",
               "content": f"Given the query {query} and background information {bg}, generate a natural conversation between a user and a system. In the conversation, the system does not have access to the background information and must ask clarifying questions to better understand and refine the query. The system should always ask clarifying questions based solely on the conversational history, without using any background information. The user knows the background information and can answer questions to clarify the query. Ensure that the user's answers are strictly based on the provided background information. The user should not introduce any new or additional information beyond what is included in the background. Use the following format:\n System: [clarifying question]\n User: [answer based on the background information]"
            }],

      max_tokens=1024,
      #do_sample=True, 
      #temperature=1.0
    )


    return completion.choices[0].message.content

In [5]:
def process_bg(data):
    data['processed_bg'] = data.apply(lambda row: row['bg'].replace(row['query'], '', 1)  if row['bg'].startswith(row['query']) else row['bg'], axis=1)
    data = data.reset_index(drop = True)
    return data

In [6]:

def generate_method3(file_path, save_filename, model):
    data = pd.read_csv(file_path, lineterminator = '\n')
    processed_data = process_bg(data)
    for i in tqdm(range(len(processed_data))):
        processed_data.loc[i, 'gen_conv'] = 'User: ' + processed_data['query'][i] + '\n' + gpt_conv(processed_data['query'][i], processed_data['processed_bg'][i], model)
    processed_data.to_csv(save_filename)
    return processed_data
    
    

In [6]:
def generate_method2(file_path, save_filename, model):
    data = pd.read_csv(file_path, lineterminator = '\n')
    processed_data = process_bg(data)
    for i in tqdm(range(len(processed_data))): 
        processed_data.loc[i, 'extracted_points'] = gpt_stage1(processed_data['query'][i], processed_data['processed_bg'][i], model, '/workspace/prompt_stage1.txt')
        processed_data.loc[i, 'gen_conv'] = gpt_stage2(processed_data['query'][i], processed_data['extracted_points'][i], model, '/workspace/prompt_stage2.txt')
        processed_data.loc[i, 'refined_conv'] = gpt_stage3('User:' + processed_data['query'][i] + '\n' + processed_data['gen_conv'][i], model, '/workspace/prompt_stage3.txt')        
    processed_data.to_csv(save_filename)
    return processed_data